# ATM E-Journal ETL - manual run

Runs the **same** ETL the Airflow DAG runs: batches of journal files -> the existing
parser on Spark -> parquet -> Greenplum -> `processed_files.csv` -> parquet cleanup.

Use it to:

* run a controlled number of batches by hand (initial load, catch-up, a re-run after a fix),
* look at what a batch produced before it goes to Greenplum (`DRY_RUN = True`),
* **write a batch into Greenplum yourself** (section 7) with the plain Spark JDBC write,
* inspect the tracking CSV, the pending markers and the run summary.

**Setup** - point `ETL_HOME` at the project root (the folder holding `config/`, `src/`,
`processed/`) and `CONFIG_PATH` at the configuration file. Nothing else is configured
here: every value - url, user, password, driver, table - comes from
`config/atm_ejournal.conf`.

## 1. Environment

In [ ]:
import json
import os
import sys

ETL_HOME = os.path.abspath(os.path.join(os.getcwd(), ".."))   # <- EDIT if the notebook is moved
CONFIG_PATH = os.path.join(ETL_HOME, "config", "atm_ejournal.conf")
ETL_NAME = "atm_ejournal"

os.environ["ATM_ETL_HOME"] = ETL_HOME
sys.path.insert(0, os.path.join(ETL_HOME, "src"))

from config_loader import list_etls, load_config
from etl_runner import AtmEjournalEtl
from file_registry import ProcessedFileRegistry, PendingBatchStore, discover_files, iter_batches

print("ETL home   :", ETL_HOME)
print("config     :", CONFIG_PATH)
print("ETLs found :", list_etls(CONFIG_PATH))

### Preflight - can this installation ship Python code to Spark?

PySpark serialises every Python function it sends to the executors with the cloudpickle
version it bundles. A PySpark older than the interpreter cannot do that, and **every**
PySpark job fails on the driver with

```
_pickle.PicklingError: Could not serialize object: IndexError: tuple index out of range
```

This cell answers it in a second (no cluster needed). If it reports FAIL, install a
PySpark that supports this Python - ideally the version the cluster runs
(`pip install "pyspark==<cluster Spark version>"`) - before going further.

In [ ]:
from check_environment import main as check_environment

check_environment([])          # add "--spark" to also run a one-task distributed job

## 2. Load and review the configuration

`safe_dump()` masks the password, so the output is safe to paste into a ticket.

In [ ]:
cfg = load_config(CONFIG_PATH, ETL_NAME)

print("input      :", cfg.path("input.INPUT_PATH"))
print("batch size :", cfg.get_int("input.BATCH_SIZE"))
print("parquet    :", cfg.path("parquet.PARQUET_PATH"))
print("tracking   :", cfg.path("tracking.PROCESSED_FILES_CSV"))
print("greenplum  :", cfg.get("greenplum.GREENPLUM_URL"),
      "->", f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}.{cfg.get('greenplum.GREENPLUM_TABLE')}",
      "| user:", cfg.get("greenplum.GREENPLUM_USER"),
      "| driver:", cfg.get("greenplum.GREENPLUM_DRIVER"))
print("strategy   :", cfg.get("greenplum.GREENPLUM_LOAD_STRATEGY"),
      "| mode:", cfg.get("greenplum.GREENPLUM_WRITE_MODE"))

print(json.dumps(cfg.safe_dump(), indent=2)[:2000])

## 3. What would this run do?

Counts the input directory and the tracking CSV without processing anything -
the same numbers the run logs and the success e-mail report.

In [ ]:
registry = ProcessedFileRegistry(cfg.path("tracking.PROCESSED_FILES_CSV"),
                                 key_mode=str(cfg.get("tracking.FILE_KEY_MODE", "path")))
processed_keys = registry.load_keys(refresh=True)
key_mode = registry.key_mode

discovered = remaining = 0
first_remaining = []
for item in discover_files(cfg.path("input.INPUT_PATH"),
                           patterns=cfg.get_list("input.FILE_PATTERN"),
                           folder_depth=cfg.get_int("input.ATM_FOLDER_DEPTH", 1),
                           min_file_age_seconds=cfg.get_int("input.MIN_FILE_AGE_SECONDS", 0)):
    discovered += 1
    if item.key(key_mode) not in processed_keys:
        remaining += 1
        if len(first_remaining) < 10:
            first_remaining.append(item.relative_path)

batch_size = cfg.get_int("input.BATCH_SIZE")
print("files discovered     :", discovered)
print("previously processed :", len(processed_keys))
print("remaining            :", remaining)
print("batch size           :", batch_size)
print("batches              :", (remaining + batch_size - 1) // batch_size)
print("next files           :", first_remaining)

## 4. Run the full ETL (parse -> parquet -> Greenplum -> tracking)

* `DRY_RUN = True` stops after the parquet write: nothing is sent to Greenplum and
  nothing is marked as processed. Use it for a first look at new journal files, then
  load those batches by hand in section 7 if you want to go step by step.
* `MAX_BATCHES` limits one manual run (`None` = every remaining file).
* `BATCH_SIZE_OVERRIDE` overrides `input.BATCH_SIZE` for this run only.

Files already in `processed_files.csv` are skipped, so re-running only picks up what is
left.

In [ ]:
# ---- EDIT ME -------------------------------------------------------------
DRY_RUN = False
MAX_BATCHES = 1            # e.g. 1 for a careful first batch, None for everything
BATCH_SIZE_OVERRIDE = None # e.g. 100
RUN_ID = None              # None -> timestamp
# --------------------------------------------------------------------------

overrides = {}
if BATCH_SIZE_OVERRIDE:
    overrides["input.BATCH_SIZE"] = BATCH_SIZE_OVERRIDE

run_cfg = load_config(CONFIG_PATH, ETL_NAME, overrides=overrides)
etl = AtmEjournalEtl(run_cfg, run_id=RUN_ID, dry_run=DRY_RUN, max_batches=MAX_BATCHES)
summary = etl.run()

print(summary.status, "|", summary.duration_human)

## 5. Result of the run

In [ ]:
interesting = ["status", "run_id", "files_discovered", "files_previously_processed",
               "files_to_process", "batches_planned", "batches_processed", "batches_failed",
               "files_processed", "files_failed", "records_processed", "records_loaded",
               "greenplum_table", "log_path", "summary_path", "stage", "batch_id", "error"]
for key in interesting:
    print(f"{key:<28}", getattr(summary, key))

print("\nper batch:")
for batch in summary.batches:
    print(f"  {batch['batch_id']:<12} files={batch['file_count']:<5} records={batch['records']:<7}"
          f" loaded={batch['rows_loaded']:<7} {batch['duration_seconds']:>6.1f}s  {batch['status']}"
          + (f"  ({batch['stage']}: {batch['error']})" if batch['status'] == 'FAILED' else ""))

print("\nparser counters:", json.dumps(summary.parser_stats, indent=2))

## 6. Inspect what a batch wrote

The parquet of a successful batch is deleted once Greenplum confirms the load, so this
shows dry-run batches and batches that failed before cleanup.

In [ ]:
from spark_session import build_spark_session, stop_spark_session

parquet_root = cfg.path("parquet.PARQUET_PATH")
batches_on_disk = sorted(os.listdir(parquet_root)) if os.path.isdir(parquet_root) else []
print("parquet directories:", batches_on_disk)

spark = None
batch_df = None
if batches_on_disk:
    BATCH_DIR = os.path.join(parquet_root, batches_on_disk[0])   # <- EDIT to pick another batch
    spark = build_spark_session(cfg)
    batch_df = spark.read.parquet(BATCH_DIR)
    print(BATCH_DIR, "->", batch_df.count(), "rows")
    batch_df.select("ATM_NO", "TRANSACTION_DATETIME", "AMOUNT", "STATUS", "DENOMINATION",
                    "NOTES_COUNT", "SOURCE_FILE", "BATCH_ID").show(20, truncate=False)
    batch_df.groupBy("ATM_NO", "STATUS").count().show()

## 7. Write to Greenplum

The write is the plain Spark JDBC write, with `url`, `dbtable`, `user`, `password` and
`driver` taken from the configuration file:

```python
df.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://<host>:5432/<database>") \
    .option("dbtable", "schema.table_name") \
    .option("user", "username") \
    .option("password", "password") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()
```

**7a** does exactly that from the notebook. **7b** does the same write through
`GreenplumLoader`, which additionally records the batch in the control table and verifies
the loaded row count - that is what the ETL itself calls, so use it when you want the
manual load to stay consistent with the automated one.

In [ ]:
# ---- 7a. the JDBC write, spelled out -------------------------------------
WRITE_MODE = cfg.get("greenplum.GREENPLUM_WRITE_MODE", "append")   # append | overwrite

if batch_df is not None:
    (batch_df.write
        .format("jdbc")
        .option("url", cfg.get("greenplum.GREENPLUM_URL"))
        .option("dbtable", f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}."
                           f"{cfg.get('greenplum.GREENPLUM_TABLE')}")
        .option("user", cfg.get("greenplum.GREENPLUM_USER"))
        .option("password", cfg.get("greenplum.GREENPLUM_PASSWORD"))
        .option("driver", cfg.get("greenplum.GREENPLUM_DRIVER"))
        .option("batchsize", cfg.get_int("greenplum.GREENPLUM_JDBC_BATCH_SIZE", 20000))
        .mode(WRITE_MODE)
        .save())
    print("written to", f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}."
                        f"{cfg.get('greenplum.GREENPLUM_TABLE')}", "| mode:", WRITE_MODE)
else:
    print("no batch parquet loaded - run section 6 first")

In [ ]:
# ---- 7b. the same write through the loader (control row + row count) ------
from greenplum_loader import GreenplumLoader

if batch_df is not None:
    loader = GreenplumLoader(cfg, spark=spark)
    result = loader.load_batch(
        batch_df,
        batch_id="MANUAL_0001",          # any id; it is stored on the control row
        run_id=summary.run_id,
        file_count=0,
        parquet_path=BATCH_DIR,
    )
    print(json.dumps(result.as_dict(), indent=2, default=str))

In [ ]:
# ---- 7c. read back what is in the target table ---------------------------
if spark is not None:
    target = (spark.read
              .format("jdbc")
              .option("url", cfg.get("greenplum.GREENPLUM_URL"))
              .option("dbtable", f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}."
                                 f"{cfg.get('greenplum.GREENPLUM_TABLE')}")
              .option("user", cfg.get("greenplum.GREENPLUM_USER"))
              .option("password", cfg.get("greenplum.GREENPLUM_PASSWORD"))
              .option("driver", cfg.get("greenplum.GREENPLUM_DRIVER"))
              .load())
    print("rows in target:", target.count())
    target.groupBy("ETL_RUN_ID", "BATCH_ID").count().orderBy("ETL_RUN_ID", "BATCH_ID").show(20)

In [ ]:
# Release the session opened in section 6 when you are done.
if spark is not None:
    stop_spark_session(spark)
    spark = None

## 8. Tracking, pending markers and logs

In [ ]:
import pandas as pd

csv_path = cfg.path("tracking.PROCESSED_FILES_CSV")
if os.path.exists(csv_path):
    tracked = pd.read_csv(csv_path)
    print(f"{len(tracked)} row(s) in {csv_path}")
    display(tracked.tail(10))
    display(tracked.groupby(["batch_id", "status"]).size().rename("files").reset_index().tail(10))
else:
    print("no tracking CSV yet:", csv_path)

pending = PendingBatchStore(cfg.path("tracking.PENDING_DIR")).list_pending()
print("\nin-flight batches:", [item["batch_id"] for item in pending] or "none")

log_dir = cfg.path("logging.LOG_PATH")
logs = sorted(os.listdir(log_dir)) if os.path.isdir(log_dir) else []
total_mb = sum(os.path.getsize(os.path.join(log_dir, name)) for name in logs) / 1024 / 1024
print(f"\n{len(logs)} log file(s), {total_mb:.1f} MB in {log_dir}")
print("\n".join(logs[-5:]))

## 9. Tail the run log

Everything the batches logged, including the failure stage and stack trace when something
went wrong.

In [ ]:
if summary.log_path and os.path.exists(summary.log_path):
    with open(summary.log_path) as handle:
        lines = handle.readlines()
    print("".join(lines[-60:]))

## 10. Recovery after a failure

The ETL is restartable - normally there is nothing to do but run it again:

| What failed | What the next run does |
| --- | --- |
| Spark / parquet write | files were never marked processed -> the batch is retried |
| Greenplum write | nothing marked -> the batch is retried |
| Crash after the Greenplum write | the pending marker + the batch control table are used to finish the tracking without loading again |
| Tracking CSV update | same as above |

`processed/pending/` shows what was in flight. To force a file to be processed again,
remove its row from `processed_files.csv` - but note that with
`GREENPLUM_LOAD_STRATEGY: append` that file's rows are appended a second time; switch the
strategy to `delete_insert_by_source_file` if a reload must replace the old rows instead.

In [ ]:
# Re-run everything that is still unprocessed (no batch limit).
# summary_full = AtmEjournalEtl(load_config(CONFIG_PATH, ETL_NAME)).run()
# print(summary_full.status, summary_full.files_processed, "file(s) processed")